In [1]:
# ============================================================
# ERIP - Internal Rating Bronze Ingestion
# Notebook: nb_ingest_internal_rating
# Purpose:
# 1. Read Internal Rating Engine source file
# 2. Validate data contract
# 3. Run data quality checks
# 4. Write Bronze Delta table
# 5. Log ingestion metadata and DQ results
# ============================================================

# ====================================================
# SECTION 1
# Pipeline Initialization
# ====================================================

from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

source_system = "Internal Rating Engine"
source_file_path = "Files/01_Bronze/internal_rating_engine/internal_rating_engine.csv"
target_table = "bronze_internal_rating_engine"
pipeline_name = "nb_ingest_internal_rating"

run_start_time = datetime.now()

print("ERIP Internal Rating Engine ingestion started")
print(f"Source system: {source_system}")
print(f"Source file: {source_file_path}")
print(f"Target table: {target_table}")

StatementMeta(, 545b8013-b9ab-4261-9dd3-2252a4560429, 3, Finished, Available, Finished, False)

ERIP Internal Rating Engine ingestion started
Source system: Internal Rating Engine
Source file: Files/01_Bronze/internal_rating_engine/internal_rating_engine.csv
Target table: bronze_internal_rating_engine


In [2]:
# ====================================================
# SECTION 2
# Read source CSV
# ====================================================

rating_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(source_file_path)
)

display(rating_df.limit(10))

print(f"Rows read: {rating_df.count()}")
print(f"Columns read: {len(rating_df.columns)}")

StatementMeta(, 545b8013-b9ab-4261-9dd3-2252a4560429, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 32893658-f55b-45b9-b956-5da551b8e9c2)

Rows read: 1000
Columns read: 19


In [3]:
# ============================================================
# SECTION 3 - DATA CONTRACT VALIDATION
# ============================================================

required_columns = [
    "rating_record_id",
    "customer_id",
    "rating_date",
    "previous_internal_grade",
    "current_internal_grade",
    "pd",
    "pd_band",
    "lgd",
    "ead",
    "expected_loss",
    "watchlist_flag",
    "scorecard_type",
    "scorecard_score",
    "model_version",
    "model_override_flag",
    "override_reason",
    "ifrs9_stage_recommendation",
    "source_system",
    "extract_date"
]

missing_columns = list(
    set(required_columns) -
    set(rating_df.columns)
)

if len(missing_columns) == 0:
    print("✓ Data Contract Validation Passed")
else:
    print("✗ Missing Columns:")
    print(missing_columns)

StatementMeta(, 545b8013-b9ab-4261-9dd3-2252a4560429, 5, Finished, Available, Finished, False)

✓ Data Contract Validation Passed


In [4]:
# ============================================================
# SECTION 4 - THREE-LAYER DATA CONTRACT VALIDATION
# Layer 1: Schema Rules
# Layer 2: Business Rules
# Layer 3: Regulatory / Banking Rules
# ============================================================

validation_results = []

def add_validation_result(layer, rule_id, rule_name, failed_count):
    status = "PASS" if failed_count == 0 else "FAIL"
    validation_results.append({
        "pipeline_name": pipeline_name,
        "source_system": source_system,
        "target_table": target_table,
        "validation_layer": layer,
        "rule_id": rule_id,
        "rule_name": rule_name,
        "failed_count": int(failed_count),
        "status": status,
        "validation_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })

# ----------------------------
# Layer 1: Schema Rules
# ----------------------------
add_validation_result(
    "Schema",
    "SCHEMA_001",
    "All required columns must be present",
    len(missing_columns)
)

# ----------------------------
# Layer 2: Business Rules
# ----------------------------
total_rows = rating_df.count()

duplicate_rating_ids = total_rows - rating_df.select("rating_record_id").distinct().count()
null_rating_ids = rating_df.filter(col("rating_record_id").isNull()).count()
null_customer_ids = rating_df.filter(col("customer_id").isNull()).count()
invalid_watchlist_flag = rating_df.filter(~col("watchlist_flag").isin("Y", "N")).count()
invalid_override_flag = rating_df.filter(~col("model_override_flag").isin("Y", "N")).count()
missing_model_version = rating_df.filter(col("model_version").isNull() | (trim(col("model_version")) == "")).count()
invalid_scorecard_score = rating_df.filter((col("scorecard_score") < 0) | (col("scorecard_score") > 1000)).count()

add_validation_result("Business", "BUS_001", "Rating record ID must be unique", duplicate_rating_ids)
add_validation_result("Business", "BUS_002", "Rating record ID must not be null", null_rating_ids)
add_validation_result("Business", "BUS_003", "Customer ID must not be null", null_customer_ids)
add_validation_result("Business", "BUS_004", "Watchlist flag must be Y or N", invalid_watchlist_flag)
add_validation_result("Business", "BUS_005", "Model override flag must be Y or N", invalid_override_flag)
add_validation_result("Business", "BUS_006", "Model version must be populated", missing_model_version)
add_validation_result("Business", "BUS_007", "Scorecard score must be between 0 and 1000", invalid_scorecard_score)

# ----------------------------
# Layer 3: Regulatory / Banking Rules
# ----------------------------
invalid_pd = rating_df.filter((col("pd") < 0) | (col("pd") > 1)).count()
invalid_lgd = rating_df.filter((col("lgd") < 0) | (col("lgd") > 1)).count()
invalid_ead = rating_df.filter(col("ead") < 0).count()
invalid_expected_loss = rating_df.filter(col("expected_loss") < 0).count()
invalid_pd_band = rating_df.filter(
    ~col("pd_band").isin("Very Low", "Low", "Moderate", "Elevated", "High", "Default")
).count()
invalid_ifrs9_recommendation = rating_df.filter(
    ~col("ifrs9_stage_recommendation").isin("Stage 1", "Stage 2", "Stage 3")
).count()
invalid_current_grade = rating_df.filter(
    ~col("current_internal_grade").isin("IG1", "IG2", "IG3", "IG4", "NIG1", "NIG2", "NIG3", "Default")
).count()
invalid_previous_grade = rating_df.filter(
    ~col("previous_internal_grade").isin("IG1", "IG2", "IG3", "IG4", "NIG1", "NIG2", "NIG3", "Default")
).count()

add_validation_result("Regulatory", "REG_001", "PD must be between 0 and 1", invalid_pd)
add_validation_result("Regulatory", "REG_002", "LGD must be between 0 and 1", invalid_lgd)
add_validation_result("Regulatory", "REG_003", "EAD must not be negative", invalid_ead)
add_validation_result("Regulatory", "REG_004", "Expected loss must not be negative", invalid_expected_loss)
add_validation_result("Regulatory", "REG_005", "PD band must be valid", invalid_pd_band)
add_validation_result("Regulatory", "REG_006", "IFRS 9 stage recommendation must be valid", invalid_ifrs9_recommendation)
add_validation_result("Regulatory", "REG_007", "Current internal grade must be valid", invalid_current_grade)
add_validation_result("Regulatory", "REG_008", "Previous internal grade must be valid", invalid_previous_grade)

validation_df = spark.createDataFrame(validation_results)

display(validation_df)

failed_validations = validation_df.filter(col("status") == "FAIL").count()

if failed_validations > 0:
    raise Exception(f"Internal Rating Engine Validation Failed: {failed_validations} validation rule(s) failed.")
else:
    print("✓ Internal Rating Engine Three-layer Validation Passed")

StatementMeta(, 545b8013-b9ab-4261-9dd3-2252a4560429, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b1397ec7-5d40-46a5-83ec-fd6225a46b53)

✓ Internal Rating Engine Three-layer Validation Passed


In [5]:
# ============================================================
# SECTION 4 - VALIDATION FRAMEWORK
# Data Contract + Data Quality + Business + Regulatory Checks
# ============================================================

# Add ingestion audit columns to source data
internal_rating_bronze_df = (
    rating_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("pipeline_name", lit(pipeline_name))
    .withColumn("bronze_load_date", current_date())
)

# Write Bronze Delta table
internal_rating_bronze_df.write.mode("overwrite").format("delta").saveAsTable("bronze_internal_rating_engine")

# Write validation results table
validation_df.write.mode("append").format("delta").saveAsTable("dq_validation_results")

print("✓ Bronze Delta table created: bronze_internal_rating_engine")
print("✓ DQ validation results written: dq_validation_results")
print(f"Rows written to Bronze: {internal_rating_bronze_df.count()}")

StatementMeta(, 545b8013-b9ab-4261-9dd3-2252a4560429, 7, Finished, Available, Finished, False)

✓ Bronze Delta table created: bronze_internal_rating_engine
✓ DQ validation results written: dq_validation_results
Rows written to Bronze: 1000


In [6]:
# ============================================================
# SECTION 5 - DATA QUALITY SUMMARY
# ============================================================

total_validation_rules = validation_df.count()
passed_validation_rules = validation_df.filter(col("status") == "PASS").count()
failed_validation_rules = validation_df.filter(col("status") == "FAIL").count()

dq_score = (passed_validation_rules / total_validation_rules) * 100

print("Data Quality Summary")
print("--------------------")
print(f"Total validation rules: {total_validation_rules}")
print(f"Passed validation rules: {passed_validation_rules}")
print(f"Failed validation rules: {failed_validation_rules}")
print(f"Data Quality Score: {dq_score}%")

StatementMeta(, 545b8013-b9ab-4261-9dd3-2252a4560429, 8, Finished, Available, Finished, False)

Data Quality Summary
--------------------
Total validation rules: 16
Passed validation rules: 16
Failed validation rules: 0
Data Quality Score: 100.0%


In [8]:
# ============================================================
# SECTION 6 - METADATA LOGGING
# ============================================================

from datetime import datetime

run_end_time = datetime.now()
execution_time_seconds = (run_end_time - run_start_time).total_seconds()

metadata = [{
    "pipeline_name": pipeline_name,
    "source_system": source_system,
    "target_table": target_table,
    "rows_processed": internal_rating_bronze_df.count(),
    "validation_rules": total_validation_rules,
    "dq_score": dq_score,
    "status": "SUCCESS",
    "run_start_time": run_start_time.strftime("%Y-%m-%d %H:%M:%S"),
    "run_end_time": run_end_time.strftime("%Y-%m-%d %H:%M:%S"),
    "execution_time_seconds": execution_time_seconds
}]

metadata_df = spark.createDataFrame(metadata)

metadata_df.write \
    .mode("append") \
    .format("delta") \
    .saveAsTable("metadata_ingestion_log")

display(metadata_df)

print("✓ Metadata successfully written")

StatementMeta(, 545b8013-b9ab-4261-9dd3-2252a4560429, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 397c54e8-0db6-4be3-9471-355a07c71dc7)

✓ Metadata successfully written
